In [ ]:
!pip install groq --quiet
import os
import json
import re
import time
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
print('Libraries ready!')

Libraries ready!


In [ ]:
from groq import Groq
API_KEY = "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"
client = Groq(api_key = API_KEY)
MODEL = "llama-3.1-8b-instant"
print(f'Groq client configured with model:{MODEL}')
print('Make sure API_KEY is replaced with your actual key!')

Groq client configured with model:llama-3.1-8b-instant
Make sure API_KEY is replaced with your actual key!


In [ ]:
def ask_llm(user_message, system_message="You are a helpful assistaant.", temperature = 1, max_tokens = 1500):
  response = client.chat.completions.create(
      model = MODEL,
      messages = [
        {
          "role": "system","content": system_message
        },
        {
          "role": "user","content": user_message
        }
      ],
      temperature = temperature,
      max_tokens = max_tokens,

  )

  return response.choices[0].message.content

text_response = ask_llm(
    "Jadeja vs Hardik, who is the best give me the percentage"
)
print('===LLM Response===')
print(text_response)

===LLM Response===
Comparing the performances of Ravindra Jadeja and Hardik Pandya can be challenging, as they play different roles in the Indian cricket team. However, I can provide you with a brief analysis of their ODI and T20 stats.

Based on their career ODI statistics:

- **Ravindra Jadeja (India, 2009-2022):**
  - Matches: 160
  - Batting Average: 32.29
  - Runs: 2,956
  - Highest Score: 100
  - Bowling Average: 34.25
  - Wickets: 147
  - Economy Rate: 5.25

- **Hardik Pandya (India, 2016-2022):**
  - Matches: 76
  - Batting Average: 34.35
  - Runs: 2,142
  - Highest Score: 106
  - Bowling Average: 46.51
  - Wickets: 52
  - Economy Rate: 9.33

Considering their batting and bowling performances, Jadeja has an edge in ODI cricket:

- **Batting Percentage (Jadeja vs Hardik):** 32.29% (Jadeja) vs 34.35% (Hardik)
- **Bowling Percentage (Jadeja vs Hardik):** 34.25% (Jadeja) vs 46.51% (Hardik)

However, when it comes to T20 cricket, Hardik Pandya has been more successful:

- **Ravindra

In [ ]:
response_etl = ask_llm(
    "List me the Franchise who won the ipl titiles, year by year, also list the teams by more no. of titles, list me the teams who didn't win a single title yet",
    system_message = "You are a senior Cricket Analyst"
)
print('Medallion +  ETL Connection')
print(response_etl)
print()
print('---Token explanation---')
print('Each work is roughly 1- 2 tokens')
print('The Model above used approximately', len(response_etl.split())*1.3,'tokens.')
print('Llama-3.1-8b context window: 8192 tokens(~6000 words per conversation)')

Medallion +  ETL Connection
Here's the list of IPL title winners, year by year:

1. 2008: Rajasthan Royals
2. 2009: Deccan Chargers
3. 2010: Mumbai Indians
4. 2011: Chennai Super Kings
5. 2012: Kolkata Knight Riders
6. 2013: Mumbai Indians
7. 2014: Kolkata Knight Riders
8. 2015: Mumbai Indians
9. 2016: Sunrisers Hyderabad
10. 2017: Mumbai Indians
11. 2018: Chennai Super Kings
12. 2019: Mumbai Indians
13. 2020: Mumbai Indians
14. 2021: Chennai Super Kings
15. 2022: Gujarat Titans

Teams with more than 2 IPL titles:

1. Mumbai Indians - 5 titles (2010, 2013, 2015, 2017, 2019, 2020)
2. Chennai Super Kings - 4 titles (2010, 2011, 2018, 2021)
3. Kolkata Knight Riders - 2 titles (2012, 2014)

Teams who haven't won a single title yet:

1. Delhi Capitals (previously known as Dehi Daredevils and Dehi Capitals)
2. Royal Challengers Bangalore
3. Sunrisers Hyderabad (although they came very close in 2016 and 2018 but lost both times in finals)
4. Lucknow Super Giants 
5. Punjab Kings 
6. Rajasthan

In [ ]:
df = pd.read_csv('/content/student_performance.csv')

dept_name = df['department'].iloc[0]


zero_shot_response = ask_llm(
    f"how many students choose computer science dept in the data set: {dept_name}"
)
print('Zero-Shot Result:')
print(zero_shot_response)
print()
ambiguous_response = ask_llm("Clean this data: ramesh kumar, 45000, mumbai")
print('Ambiguous Zero-Shot Result:')
print(ambiguous_response)
print()
print('Problem: output format is unpredictable and not machine-parseable')

Zero-Shot Result:
Since you haven't provided any data, I'll give you a general example. Please note that the actual numbers will depend on the specific dataset you're referring to.

Let's assume you have a dataset with the following information:

| Year | Computer Science | Math | Engineering |
| --- | --- | --- | --- |
| 2020 | 2000 | 1500 | 1200 |
| 2021 | 2200 | 1800 | 1500 |
| 2022 | 2500 | 2000 | 1800 |

To find the total number of students who chose the Computer Science department, we can sum the values in the "Computer Science" column.

Total students in Computer Science = 2000 + 2200 + 2500 = 6700

So, in this dataset, a total of 6700 students chose the Computer Science department.

Ambiguous Zero-Shot Result:
I'll clean this data by suggesting a few possible formats:

- Name: Ramesh Kumar (assuming Kumar is the last name)
- Salary: 45,000 (adding commas for readability)
- Location: Mumbai

Cleaned Data:
Ramesh Kumar, 45,000, Mumbai

Alternatively, we can consider different for

In [ ]:
few_shot_prompt = """
  Conver exployee text to json. Here are examples:
  Input: Ramesh Kumar, 45000, mumbai
  Output:{"name": "Ramesh Kumar", "salary": 45000, "city": "mumbai"}

  Input: priya nair, 52000, Delhi
  Output:{"name": "Priya Nair", "salary": 52000, "city": "Delhi"}
"""
few_shot_response = ask_llm(
    few_shot_prompt,
    system_message="You are an assistant that converts employee text to JSON. Only output the JSON and nothing else.",
    temperature = 0.0
)
print('Few-shot Result:')
print(few_shot_response)
print()

try:
  parsed = json.loads(few_shot_response.strip())
  print('Successfully parsed JSON')
  print(f'Name: {parsed["name"]}, Salary: {parsed["salary"]}, City: {parsed["city"]}')
except json.JSONDecodeError:
  print('Parsing failed - model added extra text')
  print('Solution: add explicit instructions in the system prompt')


Few-shot Result:
{"name": "Ramesh Kumar", "salary": 45000, "city": "mumbai"}

Successfully parsed JSON
Name: Ramesh Kumar, Salary: 45000, City: mumbai


In [ ]:
same_question = """Review the following Python code for potential issues:\n\ndef process_data(data_list):\n    result = []\n    for item in data_list:\n        result.append(item * 2)\n    return result\n\nprint(process_data([1, 2, '3']))\n"""
generic_response = ask_llm(same_question, temperature = 0.2)
print('Without Role Prompting:')
print(generic_response[:300],'...')
print()

role_response = ask_llm(
    same_question,
    system_message = "You are a senior data engineer with 10 years of production."
                      "experience. Review code critically for production readiness,"
                      "data type issues, and potential failures at scale.",
    temperature = 0.2
)
print('With Role Prompting:(Senior Data Engineer)')
print(role_response[:400],'...')
print()
print('Notice: role prompting produces more technical, actionable feedback')


Without Role Prompting:
**Code Review**

The provided Python code appears to be a simple function that takes a list of data, doubles each item in the list, and returns the resulting list. However, there are a few potential issues to consider:

1. **Data Type Inconsistency**: The function does not handle data type inconsist ...

With Role Prompting:(Senior Data Engineer)
**Code Review**

The provided Python code appears to be a simple function that takes a list of data, doubles each item, and returns the result. However, there are several potential issues that need to be addressed for production-readiness:

### 1. Data Type Issues

The function does not handle different data types within the input list. In the example usage, the input list contains an integer (`1` ...

Notice: role prompting produces more technical, actionable feedback


In [ ]:
prompt = "Give me one creative name for a data analytics startup"

print('===Temperature Experiment===')
for temp in[0.0, 0.5, 1.0]:
  response = ask_llm(prompt, temperature = temp)
  print(f'Temperature {temp}: {response.strip()}')
  time.sleep(1)

  print()
  print('Observation:')
  print('temperature = 0.0 -> Same or very similar answer every run (deterministic)')
  print('temperature = 0.5 -> some variations')
  print('temperature = 1.0 -> more creative/varied, sometimes surprising')
  print()
  print('Rule for data engineering tasks: use temperature = 0.0 or 0.1')
  print('You need CONSISTENT, PARSEABLE output - not creative variation')

===Temperature Experiment===
Temperature 0.0: Here's a creative name for a data analytics startup:

**Nexa Insights**

"Nexa" suggests connection and linkages, implying the ability to connect disparate data points and provide valuable insights. This name conveys the idea of a startup that helps businesses navigate complex data landscapes and uncover hidden patterns and trends.

Observation:
temperature = 0.0 -> Same or very similar answer every run (deterministic)
temperature = 0.5 -> some variations
temperature = 1.0 -> more creative/varied, sometimes surprising

Rule for data engineering tasks: use temperature = 0.0 or 0.1
You need CONSISTENT, PARSEABLE output - not creative variation
Temperature 0.5: Here's a creative name for a data analytics startup:

**"Nexixa"**

"Nexixa" is a combination of the words "nexus" (meaning connection or link) and "analytics" (the core focus of the startup). This name suggests that the company helps connect data points to provide valuable insights and

In [ ]:
messy_invoices  = [
    "INV-2024-0891 TECHWORLD SOLUTIONS 15TH JAN 2024 Rs. 45,000 Laptop Purchase",
    "Invoice from PRIYA ENTERPRISES dt 07-02-2024 amt:12500 for Office Cleaning Services",
    "#INV-2024-103 | arjun nair consultancy 8000 | march 15 2024 | python training" ,
    "SURESH RAO HARDWARE STORE 25000 Keyboard and Mouse accessories 2024/02/20"
    "Tax Invoice: Ananya Tech Solution | INV-897 | Data: 28-Feb-24 | Amount :INR 95,000 | Server hardware",
]
print('Messy invoices to process: ')
for i, inv in enumerate(messy_invoices,1):
  print(f'{i+1}. {inv}')
print(f'\n Total {len(messy_invoices)} invoices')



Messy invoices to process: 
2. INV-2024-0891 TECHWORLD SOLUTIONS 15TH JAN 2024 Rs. 45,000 Laptop Purchase
3. Invoice from PRIYA ENTERPRISES dt 07-02-2024 amt:12500 for Office Cleaning Services
4. #INV-2024-103 | arjun nair consultancy 8000 | march 15 2024 | python training
5. SURESH RAO HARDWARE STORE 25000 Keyboard and Mouse accessories 2024/02/20Tax Invoice: Ananya Tech Solution | INV-897 | Data: 28-Feb-24 | Amount :INR 95,000 | Server hardware

 Total 4 invoices


In [ ]:
EXTRACTION_SYSTEM_PROMPT ="""You are an AI assistant specialized in extracting key information from messy invoice texts.Your task is to parse the provided invoice text and extract the following fields into a JSON object:invoice_id: The invoice number (e.g., INV-2024-0891). It might start with # or Tax Invoice.company_name: The name of the company issuing the invoice.issue_date: The date the invoice was issued (e.g., 15th JAN 2024, 07-02-2024, 2024/02/20, 28-Feb-24).amount: The total amount of the invoice. It might be preceded by 'Rs.', 'amt:', 'INR', or just be a number.description: A brief description of the goods or services.Ensure the output is a valid JSON object.If a field is not found, use `null` as its value."""
print('Extraction prompt engineered!')
print(f'System prompt length: {len(EXTRACTION_SYSTEM_PROMPT)} characters')
print(f' = {len(EXTRACTION_SYSTEM_PROMPT.split())} words, = {int(len(EXTRACTION_SYSTEM_PROMPT.split())*1.3)} tokens')

Extraction prompt engineered!
System prompt length: 696 characters
 = 109 words, = 141 tokens


In [ ]:
few_shot_name_salary_prompt = """
  Convert employee text to JSON, extracting only 'name' and 'salary'. Here are examples:
  Input: John Doe, 60000 USD
  Output: {"name": "John Doe", "salary": 60000}

  Input: Jane Smith, 75000
  Output: {"name": "Jane Smith", "salary": 75000}

  Input: Alice Brown, 92000 GBP
  Output: {"name": "Alice Brown", "salary": 92000}

  Input: David Lee, 55000
  Output: {"name": "David Lee", "salary": 55000}
"""

name_salary_response = ask_llm(
    few_shot_name_salary_prompt + "Input: Bob Johnson, 80000",
    system_message="You are an assistant that converts employee text to JSON. Only output the JSON and nothing else.",
    temperature = 0.0
)

print('Few-shot Name and Salary Result:')
print(name_salary_response)

try:
  parsed_name_salary = json.loads(name_salary_response.strip())
  print('Successfully parsed JSON')
  print(f'Name: {parsed_name_salary["name"]}, Salary: {parsed_name_salary["salary"]}')
except json.JSONDecodeError:
  print('Parsing failed - model added extra text')
  print('Solution: add explicit instructions in the system prompt')

Few-shot Name and Salary Result:
{"name": "Bob Johnson", "salary": 80000}
Successfully parsed JSON
Name: Bob Johnson, Salary: 80000


In [ ]:
invoices_df = pd.DataFrame(extraced_records)
invoices_df['amount'] = pd.to_numeric(invoices_df['amount'], errors='coerce')
invoices_df['issue_date' ] = pd.to_datetime(
    invoices_df['issue_date'], errors = 'coerce'
)
print('===SMART DATA CLEANER OUTPUT===')
print(f'Rows:{len(invoices_df)} | Columns:{len(invoices_df.columns)}')
print()
print(invoices_df.to_string(index=False))

NameError: name 'extraced_records' is not defined